The code is based on the `testing_lsh` notebook we saw during the 3rd tutorial. It has been expanded and adapted to suit the needs of the project.

## 1. Imports & Configuration

In [1]:
import os, glob, re, xml.etree.ElementTree as ET
import json
from pathlib import Path
from itertools import combinations
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_auc_score, roc_curve)

# ── Dynamic path configuration ────────────────────────────────────────────────
project_root = Path.cwd()
default_data_dir = project_root / "data"

env_data_dir = os.getenv("DATA_DIR")
local_cfg_path = project_root / "paths.local.json"

if env_data_dir:
    data_dir = Path(env_data_dir).expanduser()
elif local_cfg_path.exists():
    data_dir = Path(json.loads(local_cfg_path.read_text())["data_dir"]).expanduser()
else:
    data_dir = default_data_dir

CORPUS_DIR = data_dir / "corpus"
GT_PATH    = data_dir / "ground_truth.tsv"

np.random.seed(42)
print("Imports OK")

Imports OK


## 2. Load Ground Truth

In [2]:
gt = pd.read_csv(GT_PATH, sep="\t")
gt.columns = gt.columns.str.strip()

# sort by doc_name to match the order of files in CORPUS_DIR
gt = gt.sort_values("doc_name").reset_index(drop=True)

# remove duplicate doc_name entries if any (keep the first occurrence)
gt = gt.drop_duplicates(subset="doc_name", keep="first").reset_index(drop=True)

# crop doc name to just the the ID after "document" (e.g., "0001" from "suspicious-document0001")
gt["doc_name"] = gt["doc_name"].str.extract(r"document(\d+)")

print(f"Ground truth shape: {gt.shape}")
gt.head(10)


Ground truth shape: (902, 2)


,doc_name,plagiarism
0,00014,1
1,00044,1
2,00052,0
3,00060,0
4,00072,0
5,00083,1
6,00085,1
7,00087,0
8,00088,1
9,00093,1


In [3]:
print(gt["plagiarism"].value_counts().rename({1: "Plagiarised", 0: "Clean"}))

plagiarism
Plagiarised    451
Clean          451
Name: count, dtype: int64


## 3. Parse XML Metadata


In [4]:
def parse_xml_meta(xml_path: Path) -> dict:
    """Return a dict with doc_name, is_plagiarised, and a list of plagiarism features.
    
    The XML reference attribute points to the companion .txt file, e.g.:
        reference="suspicious-document05744.txt"
    Doc names are normalised to the stem without extension.
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()
    doc_ref  = root.get("reference", xml_path.stem + ".txt")
    doc_name = Path(doc_ref).stem   # e.g. "suspicious-document05744"

    features = []
    for feat in root.findall("feature"):
        if feat.get("name") == "artificial-plagiarism":
            features.append({
                "obfuscation"  : feat.get("obfuscation", "none"),
                "this_offset"  : int(feat.get("this_offset", 0)),
                "this_length"  : int(feat.get("this_length", 0)),
                "source_ref"   : feat.get("source_reference", ""),
                "source_offset": int(feat.get("source_offset", 0)),
                "source_length": int(feat.get("source_length", 0)),
            })

    return {"doc_name": doc_name, "is_plagiarised": len(features) > 0, "features": features}

# All XML files are in the single corpus directory
xml_files = sorted(CORPUS_DIR.glob("suspicious-document*.xml"))
print(f"Found {len(xml_files)} XML metadata files")

meta_records = [parse_xml_meta(p) for p in xml_files]
meta_df = pd.DataFrame([{k: v for k, v in r.items() if k != "features"} for r in meta_records])
# change is_plagiarised to match the GT labels (1 for plagiarised, 0 for clean)
meta_df["is_plagiarised"] = meta_df["is_plagiarised"].astype(int)
# crop doc name to just the the ID after "document" (e.g., "0001" from "suspicious-document0001")
meta_df["doc_name"] = meta_df["doc_name"].str.extract(r"document(\d+)")

print(meta_df["is_plagiarised"].value_counts().rename({True: "Plagiarised", False: "Clean"}))
meta_df.head()


Found 901 XML metadata files
is_plagiarised
Clean          451
Plagiarised    450
Name: count, dtype: int64


,doc_name,is_plagiarised
0,00014,1
1,00044,1
2,00052,0
3,00060,0
4,00072,0


# 4. Sanity Check

In [5]:
# Find plagiarised documents present in ground truth but missing from the XML metadata
gt_plag = set(gt.loc[gt["plagiarism"] == 1, "doc_name"])
meta_plag = set(
    meta_df.loc[meta_df["is_plagiarised"] == 1, "doc_name"].str.replace("suspicious-", "", regex=False)
)

missing_from_xml = sorted(gt_plag - meta_plag)
extra_in_xml = sorted(meta_plag - gt_plag)

print(f"Plagiarised docs in GT   : {len(gt_plag)}")
print(f"Plagiarised docs in XML  : {len(meta_plag)}")
print(f"Missing from XML metadata: {len(missing_from_xml)}")
print(f"Extra in XML metadata    : {len(extra_in_xml)}")

if missing_from_xml:
    print("\nMissing document(s):")
    for doc in missing_from_xml:
        print(doc, "-> expected XML:", CORPUS_DIR / f"suspicious-{doc}.xml")

        # drop the missing doc from GT to keep the dataset consistent
        gt = gt[gt["doc_name"] != doc]
        print(f"Dropped {doc} from GT, new GT shape: {gt.shape}")

Plagiarised docs in GT   : 451
Plagiarised docs in XML  : 450
Missing from XML metadata: 1
Extra in XML metadata    : 0

Missing document(s):
06862 -> expected XML: /Users/vitto/Desktop/SP26/DA/project_1/15_Suspicious_Passages/31.Suspicious_Passages/corpus/suspicious-06862.xml
Dropped 06862 from GT, new GT shape: (901, 2)


In [6]:
import xml.etree.ElementTree as ET
from collections import defaultdict

def build_plagiarism_map(xml_dir):
    """
    Build a mapping of document IDs to their plagiarized sources.
    
    Args:
        xml_dir: Path to directory containing suspicious-document*.xml files
        
    Returns:
        Dict[str, List[str]]: Maps document IDs to list of source document IDs they plagiarized from.
                              Documents with 0 plagiarism are not included in the map.
    """
    plagiarism_map = defaultdict(list)
    
    # Get all XML files for suspicious documents
    xml_files = sorted(xml_dir.glob("suspicious-document*.xml"))
    
    for xml_file in xml_files:
        # Extract document ID from filename (e.g., "suspicious-document00925.xml" -> "00925")
        doc_id = xml_file.stem.replace("suspicious-document", "")
        
        try:
            tree = ET.parse(xml_file)
            root = tree.getroot()
            
            # Find all artificial-plagiarism features
            for feature in root.findall(".//feature[@name='artificial-plagiarism']"):
                source_ref = feature.get("source_reference")
                if source_ref:
                    # Extract source document ID from reference
                    # e.g., "source-document02721.txt" -> "02721"
                    source_id = source_ref.replace("source-document", "").replace(".txt", "")
                    plagiarism_map[doc_id].append(source_id)
        except ET.ParseError as e:
            print(f"Error parsing {xml_file}: {e}")
    
    return dict(plagiarism_map)

# Build the plagiarism map
plagiarism_map = build_plagiarism_map(CORPUS_DIR)

# Print summary statistics
total_plagiarized = len(plagiarism_map)
total_docs_in_corpus = len(list(CORPUS_DIR.glob("suspicious-document*.xml")))
total_sources = sum(len(sources) for sources in plagiarism_map.values())

In [7]:
print(f"Total documents in corpus: {total_docs_in_corpus}")
print(f"Documents with plagiarism: {total_plagiarized}")
print(f"Total plagiarism instances: {total_sources}")
print(f"\nExample entries:")
for i, (doc_id, sources) in enumerate(list(plagiarism_map.items())[:5]):
    print(f"  Document {doc_id}: plagiarized from {sources}")

Total documents in corpus: 901
Documents with plagiarism: 450
Total plagiarism instances: 4676

Example entries:
  Document 00014: plagiarized from ['03390', '05760', '04633', '05507', '02701', '00852', '03686', '04633', '04678', '03390', '05822', '06355', '04633', '03686', '03686', '02701', '04633', '03686', '03686', '03686', '03686', '03686', '04633', '05822', '00912']
  Document 00044: plagiarized from ['05976', '05976', '05976', '05976', '05976', '05976']
  Document 00083: plagiarized from ['04661', '04886', '03574', '04886', '04886', '05659', '03574', '02104', '03574', '04023', '03574', '01167', '03574', '03574', '03574', '03574', '04661', '04886', '03574', '01167', '02104']
  Document 00085: plagiarized from ['05039', '05039', '05039']
  Document 00088: plagiarized from ['07074', '05235', '07074', '07074', '05235']


In [8]:
# sanity check: check that the keys in the plagiarism map match the entries with a 1 in the GT
assert set(gt.loc[gt["plagiarism"] == 1, "doc_name"]) == set(plagiarism_map.keys()), "Mismatch between GT and XML metadata for plagiarised documents"
# # if there is a mismatch, print the offending document ID and the expected XML file path
# for doc_id in gt.loc[gt["plagiarism"] == 1, "doc_name"]:
#     if doc_id not in plagiarism_map:
#         print(f"GT indicates {doc_id} is plagiarised but it's missing from XML metadata.")
#         print(f"Expected XML file: {CORPUS_DIR / f'suspicious-{doc_id}.xml'}")

## 5. Pre-processing

### 5.1 Load suspicious document texts


In [9]:
def load_corpus(paths) -> dict:
    """Load a list of .txt Paths → {stem: normalised_text}."""
    corpus = {}
    for p in paths:
        text = p.read_text(encoding="utf-8", errors="ignore")
        text = re.sub(r"\s+", " ", text.strip().lower())
        corpus[p.stem] = text
    return corpus

# load a single doc (saves time and memory)
def load_single_doc(path) -> str:
    text = path.read_text(encoding="utf-8", errors="ignore")
    text = re.sub(r"\s+", " ", text.strip().lower())
    return text

txts = sorted(CORPUS_DIR.glob("source-document*.txt"))
corpus = load_corpus(txts)

print(f"Suspicious docs loaded : {len(corpus)}")


Suspicious docs loaded : 902


### 5.2 Shingling

In [10]:
K_SHINGLE = 9   # character k-gram size; 8-10 is typical for paragraph-level near-duplicates

# def build_shingles(text: str, k: int = K_SHINGLE) -> frozenset:
#     """Return the set of all k-character substrings of *text*."""
#     return frozenset(text[i:i+k] for i in range(len(text) - k + 1))

def build_shingles(sentence: str, k: int):
   # Convert a sentence → set of substrings (shingles)
    shingles = []
    for i in range(len(sentence) - k):
        shingles.append(sentence[i:i+k])
    return set(shingles)

shingles = []
for name, text in corpus.items():
    shingles.append(build_shingles(text, K_SHINGLE))
# shingles = {name: build_shingles(text, K_SHINGLE) for name, text in corpus.items()}
print(f"Number of shingle sets: {len(shingles)}")


Number of shingle sets: 902


### 5.3 Build Global Vocabulary & One-Hot Encoding

In [11]:
from scipy.sparse import csr_matrix

def build_vocab(shingle_sets: list):
    # Convert list of shingle sets into a global shingle->index map.
    full_set = {item for set_ in shingle_sets for item in set_}
    vocab = {}
    for i, shingle in enumerate(full_set):
        vocab[shingle] = i
    return vocab

def shingle_index_list(shingle_set: set, vocab: dict):
    # Return vocab indices for all shingles present in one document.
    return [vocab[s] for s in shingle_set if s in vocab]

In [ ]:
# build vocab
vocab = build_vocab(shingles)
print(f"Vocab size: {len(vocab):,}")

# Build sparse one-hot matrix directly in CSR format.
rows = []
cols = []

for row_idx, shingle_set in enumerate(shingles):
    idxs = shingle_index_list(shingle_set, vocab)
    rows.extend([row_idx] * len(idxs))
    cols.extend(idxs)

data = np.ones(len(rows), dtype=np.uint8)
shingles_1hot = csr_matrix(
    (data, (np.array(rows, dtype=np.int32), np.array(cols, dtype=np.int32))),
    shape=(len(shingles), len(vocab)),
    dtype=np.uint8,
)

print(f"Sparse one-hot shape: {shingles_1hot.shape}")
print(f"Non-zeros: {shingles_1hot.nnz:,}")
print(f"Density: {shingles_1hot.nnz / (shingles_1hot.shape[0] * shingles_1hot.shape[1]):.8f}")

In [ ]:
# all_shingles: set = set()
# for shingle_set in shingles.values():
#     all_shingles.update(shingle_set)

# vocab = {shingle: idx for idx, shingle in enumerate(sorted(all_shingles))}
# VOCAB_SIZE = len(vocab)
# print(f"Vocabulary size: {VOCAB_SIZE:,}")

# def one_hot_vector(shingles: frozenset, vocab: dict) -> np.ndarray:
#     vec = np.zeros(len(vocab), dtype=np.uint8)
#     for s in shingles:
#         if s in vocab:
#             vec[vocab[s]] = 1
#     return vec

# doc_names = sorted(shingles.keys())   # e.g. ["suspicious-document00001", ...]
# print(f"Building one-hot matrix for {len(doc_names)} documents …")
# one_hot_matrix = np.stack([one_hot_vector(shingles[n], vocab) for n in doc_names])
# print(f"One-hot matrix shape: {one_hot_matrix.shape}")


In [ ]:
# print the first 10 elements of the vocab to see what the shingles look like
print("First 10 shingles in vocab:")
for i, shingle in enumerate(vocab.keys()):
    if i >= 10:
        break
    print(f"{i}: '{shingle}'")

In [ ]:
shingles_1hot[:5]

## 6. MinHash Signatures

In [ ]:
def minhash_arr(vocab: dict, resolution: int):
    # Goal: Create a MinHash matrix with random permutations
    # Each row represents a different hash function (permutation of indices)
    length = len(vocab.keys())
    arr = np.zeros((resolution, length))
    for i in range(resolution):
        permutation = np.random.permutation(len(vocab)) + 1
        arr[i, :] = permutation.copy()
    return arr.astype(int)

def get_signature(minhash, vector):
    # Goal: Compute the MinHash signature for a given one-hot vector
    # This produces a compact representation for similarity comparison
    # get index locations of every 1 value in vector
    idx = np.nonzero(vector)[0].tolist()
    # use index locations to pull only +ve positions in minhash
    shingles = minhash[:, idx]
    # find minimum value in each hash vector
    signature = np.min(shingles, axis=1)
    return signature

In [ ]:
n_hashes = 100
arr = minhash_arr(vocab, n_hashes)
# Goal: Generate 100 MinHash functions (random permutations) based on the vocabulary
signatures = []

for vector in shingles_1hot:
    signatures.append(get_signature(arr, vector))
# Goal: Compute a compact MinHash signature for each one-hot encoded sentence

# merge signatures into single array
signatures = np.stack(signatures)
# Goal: Combine all signatures into one matrix (each row = one sentence signature)
signatures.shape

In [ ]:
# N_HASH = 128   # number of hash functions (signature length)

# def build_minhash_matrix(vocab_size: int, n_hash: int) -> np.ndarray:
#     """Return (n_hash × vocab_size) matrix of random permutations."""
#     mat = np.zeros((n_hash, vocab_size), dtype=np.int32)
#     for i in range(n_hash):
#         mat[i] = np.random.permutation(vocab_size) + 1   # 1-indexed
#     return mat

# minhash_mat = build_minhash_matrix(VOCAB_SIZE, N_HASH)

# def compute_signature(minhash_mat: np.ndarray, one_hot_vec: np.ndarray) -> np.ndarray:
#     """Return the MinHash signature for a single one-hot document vector."""
#     active_cols = np.nonzero(one_hot_vec)[0]
#     return minhash_mat[:, active_cols].min(axis=1)

# print("Computing signatures …")
# signatures = np.stack([compute_signature(minhash_mat, vec) for vec in one_hot_matrix])
# print(f"Signature matrix shape: {signatures.shape}  (docs × hash functions)")


## 7. Locality-Sensitive Hashing (LSH)


In [ ]:
from itertools import combinations

class LSH:
    buckets = []
    counter = 0

    def __init__(self, b):
        # Goal: Initialize the LSH structure with b bands
        # Each band will store buckets of similar signature parts
        
        self.b = b
        for i in range(b):
            self.buckets.append({})

    def make_subvecs(self, signature):
        # Goal: Split a MinHash signature into b equal-sized subvectors (bands)
        # Each band will be used to group similar signatures
        
        l = len(signature)
        assert l % self.b == 0
        r = int(l / self.b)
        
        # break signature into subvectors
        subvecs = []
        for i in range(0, l, r):
            subvecs.append(signature[i:i+r])
        
        return np.stack(subvecs)
    
    def add_hash(self, signature):
        # Goal: Hash each band of the signature into buckets
        # Similar signatures will fall into the same bucket in at least one band
        
        subvecs = self.make_subvecs(signature).astype(str)
        
        for i, subvec in enumerate(subvecs):
            subvec = ','.join(subvec)
            
            if subvec not in self.buckets[i].keys():
                self.buckets[i][subvec] = []
            
            self.buckets[i][subvec].append(self.counter)
        
        self.counter += 1
        # Goal: Assign a unique ID to each signature added

    def check_candidates(self):
        # Goal: Find candidate pairs of similar items
        # Items that share a bucket in any band are considered potential matches
        
        candidates = []
        
        for bucket_band in self.buckets:
            keys = bucket_band.keys()
            
            for bucket in keys:
                hits = bucket_band[bucket]
                
                if len(hits) > 1:
                    candidates.extend(combinations(hits, 2))
        
        return set(candidates)
        # Goal: Return unique pairs of candidate similar items

In [ ]:
b = 20
# Goal: Define the number of bands for LSH

lsh = LSH(b)
# Goal: Initialize the LSH model with b bands (each band will group similar signatures)

for signature in signatures:
    lsh.add_hash(signature)
# Goal: Insert each MinHash signature into LSH buckets to group similar items together

In [ ]:
candidate_pairs = lsh.check_candidates()
# Goal: Retrieve all candidate pairs of sentences that were placed in the same bucket
# in at least one band → these are likely similar pairs (filtered by LSH)

len(candidate_pairs)
# Goal: Count how many candidate pairs were found (reduces comparisons vs all possible pairs)

In [ ]:
list(candidate_pairs)[:5]

In [ ]:
# TOOD